In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout, BatchNormalization

# Load the dataset
data = pd.read_csv("E:\\Codding work\\\DataForDataScienceAndAI\\Cars_data.csv")


<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:12: SyntaxWarning: invalid escape sequence '\D'
C:\Users\narut\AppData\Local\Temp\ipykernel_1452\4115007821.py:12: SyntaxWarning: invalid escape sequence '\D'
  data = pd.read_csv("E:\\Codding work\\\DataForDataScienceAndAI\\Cars_data.csv")


In [2]:

# Step 1: Fix formatting issues
# Price (`pu`)
data['pu'] = data['pu'].str.replace(',', '').astype(float)

# Max Power
data['Max Power'] = data['Max Power'].str.extract(r'(\d+\.?\d*)bhp').astype(float)

# Max Torque
data['Max Torque'] = data['Max Torque'].str.extract(r'(\d+\.?\d*)Nm').astype(float)

# Top Speed
data['Top Speed'] = data['Top Speed'].str.extract(r'(\d+\.?\d*)').astype(float)

# Acceleration
data['Acceleration'] = data['Acceleration'].str.extract(r'(\d+\.?\d*)').astype(float)

# Mileage (`mileage_new`)
data['mileage_new'] = data['mileage_new'].str.extract(r'(\d+\.?\d*)').astype(float)



In [3]:
# Turbo Charger and Super Charger
data['Turbo Charger'] = data['Turbo Charger'].str.lower().map({'yes': 1, 'no': 0})
data['Super Charger'] = data['Super Charger'].str.lower().map({'yes': 1, 'no': 0})


In [4]:

# Step 4: Balance the dataset
# Define features to use
features = ["pu", "Max Power", "Max Torque", "Top Speed", "Acceleration", "mileage_new", "Turbo Charger", "Super Charger", "km_driven"]
X = data[features]
y = data['tt']
# Encode target variable (`tt`)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)



# Step 5: Split the data


In [5]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.preprocessing import StandardScaler


# Scale features (fit on training, transform both sets)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


# Impute missing values (fit on training, transform both sets)
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train_scaled)
X_val_imputed = imputer.transform(X_val_scaled)


# Apply SMOTE only to the training set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_imputed, y_train)


In [6]:

# Step 6: Build the neural network
model = Sequential([
    Dense(64, input_dim=X_train_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


c:\Users\narut\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Train the model with full-batch gradient descent (no mini-batches)
history = model.fit(
    X_train_res, y_train_res,
    epochs=100,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.4220 - loss: 0.9929 - val_accuracy: 0.6781 - val_loss: 0.6701
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.4514 - loss: 0.9082 - val_accuracy: 0.6880 - val_loss: 0.6665
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4876 - loss: 0.8348 - val_accuracy: 0.6974 - val_loss: 0.6627
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 760ms/step - accuracy: 0.5350 - loss: 0.7738 - val_accuracy: 0.7110 - val_loss: 0.6588
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - accuracy: 0.5909 - loss: 0.7240 - val_accuracy: 0.7282 - val_loss: 0.6548
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 515ms/step - accuracy: 0.6253 - loss: 0.6838 - val_accuracy: 0.7431 - val_loss: 0.6511
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step - accuracy: 0.6525 - loss: 0.6516 - val_accuracy: 0.7501 - val_loss: 0.6475
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 655ms/step - accuracy: 0.6788 - loss: 0.6256 - val_accuracy: 0.7600 - val_l

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on validation set
y_pred_prob = model.predict(X_val_imputed)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [8]:
from tensorflow.keras.layers import Dense, Dropout
# Model 2 
model2 = Sequential([
    Dense(64, input_dim=X_train_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(16, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(1, activation='sigmoid')  # Binary classification
])

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [9]:
# Train the model with full-batch gradient descent (no mini-batches)
history2 = model2.fit(
    X_train_res, y_train_res,
    epochs=250,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model2.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step - accuracy: 0.4912 - loss: 0.8561 - val_accuracy: 0.5754 - val_loss: 0.6827
Epoch 2/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5243 - loss: 0.8069 - val_accuracy: 0.5934 - val_loss: 0.6782
Epoch 3/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 0.5583 - loss: 0.7690 - val_accuracy: 0.6163 - val_loss: 0.6739
Epoch 4/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step - accuracy: 0.5811 - loss: 0.7302 - val_accuracy: 0.6267 - val_loss: 0.6697
Epoch 5/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step - accuracy: 0.6103 - loss: 0.7048 - val_accuracy: 0.6346 - val_loss: 0.6656
Epoch 6/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.6366 - loss: 0.6758 - val_accuracy: 0.6437 - val_loss: 0.6617
Epoch 7/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 654ms/step - accuracy: 0.6529 - loss: 0.6559 - val_accuracy: 0.6486 - val_loss: 0.6580
Epoch 8/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.6708 - loss: 0.6399 - val_accuracy: 0.6540 - val_l

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on validation set
y_pred_prob = model2.predict(X_val_imputed)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
from tensorflow.keras.layers import Dense, Dropout
# Model 3
model3 = Sequential([
    Dense(128, input_dim=X_train_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(1, activation='sigmoid')  # Binary classification
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


c:\Users\narut\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Train the model with full-batch gradient descent (no mini-batches)
history3 = model3.fit(
    X_train_res, y_train_res,
    epochs=350,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model3.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step - accuracy: 0.4907 - loss: 0.8998 - val_accuracy: 0.4748 - val_loss: 0.6996
Epoch 2/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5960 - loss: 0.7266 - val_accuracy: 0.5190 - val_loss: 0.6939
Epoch 3/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 635ms/step - accuracy: 0.6722 - loss: 0.6224 - val_accuracy: 0.5380 - val_loss: 0.6890
Epoch 4/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7230 - loss: 0.5609 - val_accuracy: 0.5509 - val_loss: 0.6848
Epoch 5/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7428 - loss: 0.5314 - val_accuracy: 0.5650 - val_loss: 0.6814
Epoch 6/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - accuracy: 0.7523 - loss: 0.5106 - val_accuracy: 0.5732 - val_loss: 0.6787
Epoch 7/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 477ms/step - accuracy: 0.7558 - loss: 0.5019 - val_accuracy: 0.5878 - val_loss: 0.6768
Epoch 8/350
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 586ms/step - accuracy: 0.7595 - loss: 0.4951 - val_accuracy: 0.6013 - val_loss: 0

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on validation set
y_pred_prob = model3.predict(X_val_imputed)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
from tensorflow.keras.layers import Dense, Dropout
# Model 4 
model4 = Sequential([
    Dense(128, input_dim=X_train_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')  # Binary classification
])

model4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the model with full-batch gradient descent (no mini-batches)
history4 = model4.fit(
    X_train_res, y_train_res,
    epochs=250,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model4.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - accuracy: 0.5994 - loss: 0.6941 - val_accuracy: 0.7632 - val_loss: 0.6538
Epoch 2/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6478 - loss: 0.6458 - val_accuracy: 0.7458 - val_loss: 0.6504
Epoch 3/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 449ms/step - accuracy: 0.6833 - loss: 0.6103 - val_accuracy: 0.7350 - val_loss: 0.6484
Epoch 4/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 842ms/step - accuracy: 0.7088 - loss: 0.5828 - val_accuracy: 0.7285 - val_loss: 0.6475
Epoch 5/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 837ms/step - accuracy: 0.7209 - loss: 0.5686 - val_accuracy: 0.7241 - val_loss: 0.6475
Epoch 6/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 886ms/step - accuracy: 0.7319 - loss: 0.5519 - val_accuracy: 0.7212 - val_loss: 0.6481
Epoch 7/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7400 - loss: 0.5393 - val_accuracy: 0.7143 - val_loss: 0.6493
Epoch 8/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 774ms/step - accuracy: 0.7400 - loss: 0.5324 - val_accuracy: 0.7069 - val_loss

In [ ]:
from tensorflow.keras.layers import Dense, Dropout
# Model 5
model5 = Sequential([
    Dense(32, input_dim=X_train_res.shape[1], activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')  # Binary classification
])

model5.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the model with full-batch gradient descent (no mini-batches)
history5 = model5.fit(
    X_train_res, y_train_res,
    epochs=100,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model5.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.5375 - loss: 0.6735 - val_accuracy: 0.3792 - val_loss: 0.7193
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 597ms/step - accuracy: 0.5496 - loss: 0.6650 - val_accuracy: 0.4008 - val_loss: 0.7143
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.5633 - loss: 0.6585 - val_accuracy: 0.4287 - val_loss: 0.7095
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5743 - loss: 0.6529 - val_accuracy: 0.4691 - val_loss: 0.7050
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.5893 - loss: 0.6473 - val_accuracy: 0.5173 - val_loss: 0.7008
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.6029 - loss: 0.6381 - val_accuracy: 0.5593 - val_loss: 0.6968
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step - accuracy: 0.6155 - loss: 0.6336 - val_accuracy: 0.5865 - val_loss: 0.6930
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step - accuracy: 0.6243 - loss: 0.6300 - val_accuracy: 0.6092 - val_los

In [ ]:
from tensorflow.keras.layers import Dense, Dropout
# Model 6
model6 = Sequential([
    Dense(128, input_dim=X_train_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Binary classification
])

model6.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


c:\Users\narut\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Train the model with full-batch gradient descent (no mini-batches)
history6 = model6.fit(
    X_train_res, y_train_res,
    epochs=250,
    batch_size=X_train_res.shape[0],  # Use full dataset as a single batch
    validation_data=(X_val_imputed, y_val),
    verbose=1
)

# Evaluate
val_loss, val_accuracy = model6.evaluate(X_val_imputed, y_val, verbose=0)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

Epoch 1/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 19s 19s/step - accuracy: 0.5697 - loss: 0.8110 - val_accuracy: 0.4723 - val_loss: 0.7039
Epoch 2/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.6106 - loss: 0.7418 - val_accuracy: 0.5105 - val_loss: 0.6956
Epoch 3/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 751ms/step - accuracy: 0.6463 - loss: 0.6800 - val_accuracy: 0.5350 - val_loss: 0.6885
Epoch 4/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 565ms/step - accuracy: 0.6728 - loss: 0.6402 - val_accuracy: 0.5666 - val_loss: 0.6828
Epoch 5/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 765ms/step - accuracy: 0.6892 - loss: 0.6112 - val_accuracy: 0.5948 - val_loss: 0.6778
Epoch 6/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7060 - loss: 0.5846 - val_accuracy: 0.6186 - val_loss: 0.6734
Epoch 7/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 818ms/step - accuracy: 0.7152 - loss: 0.5691 - val_accuracy: 0.6332 - val_loss: 0.6697
Epoch 8/250
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7253 - loss: 0.5588 - val_accuracy: 0.6515 - val_loss: 0

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on validation set
y_pred_prob = model6.predict(X_val_imputed)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Accuracy: 0.7996
Precision: 0.9385
Recall: 0.7902
